# Model Evaluation — 3 charts cho báo cáo BGK

Chạy SAU step 7. Sinh ra:
1. **Correlation Heatmap** — chứng minh 60 features đã chọn lọc có hệ thống
2. **SHAP Summary Plot** — giải thích chiều hướng tác động (interpretability)
3. **Score Distribution Cold vs Warm** — chứng minh model phân biệt 2 cohort

Output: PNG saved to `model/figs/`

In [ ]:
import os, sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import lightgbm as lgb
from config import CACHE_DIR, MODEL_DIR

FIGS_DIR = os.path.join(MODEL_DIR, 'figs')
os.makedirs(FIGS_DIR, exist_ok=True)
print('Saving figures to:', FIGS_DIR)

## Load data & model

In [ ]:
train_df = pd.read_parquet(f'{CACHE_DIR}/features_train.parquet')
ranked   = pd.read_parquet(f'{CACHE_DIR}/ranked_predictions.parquet')
model    = lgb.Booster(model_file=f'{CACHE_DIR}/lgbm_ranker.txt')

FEATURE_COLS = [c for c in train_df.columns if c not in {'user_id','item_id','label'}]
print(f'Train rows: {len(train_df):,}  Features: {len(FEATURE_COLS)}')
print(f'Ranked rows: {len(ranked):,}  Users: {ranked["user_id"].nunique():,}')

## Chart 1 — Correlation Heatmap

In [ ]:
sample = train_df.sample(n=min(500_000, len(train_df)), random_state=42)
corr = sample[FEATURE_COLS].corr()

plt.figure(figsize=(20, 18))
sns.heatmap(corr, cmap='RdBu_r', center=0, vmin=-1, vmax=1,
            cbar_kws={'shrink': 0.6}, square=True, linewidths=0.1)
plt.title('Feature Correlation Heatmap (60 features, sample 500K rows)', fontsize=14)
plt.tight_layout()
plt.savefig(f'{FIGS_DIR}/01_correlation_heatmap.png', dpi=110, bbox_inches='tight')
plt.show()
print('Saved: 01_correlation_heatmap.png')

## Chart 2 — SHAP Summary Plot (Top-20 features)

In [ ]:
import shap

X_sample = sample[FEATURE_COLS].astype(np.float32).replace([np.inf, -np.inf], np.nan)
X_sample = X_sample.sample(n=10_000, random_state=42)

explainer = shap.TreeExplainer(model)
shap_values = explainer.shap_values(X_sample)

plt.figure(figsize=(10, 8))
shap.summary_plot(shap_values, X_sample, feature_names=FEATURE_COLS,
                  max_display=20, show=False)
plt.title('SHAP Summary — Top 20 Features Impact', fontsize=13)
plt.tight_layout()
plt.savefig(f'{FIGS_DIR}/02_shap_summary.png', dpi=110, bbox_inches='tight')
plt.show()
print('Saved: 02_shap_summary.png')

## Chart 3 — Score Distribution: Cold vs Warm Users

Cold user = user không có lịch sử trong train pos. Warm = user có lịch sử.

In [ ]:
pos = pd.read_parquet(f'{CACHE_DIR}/user_item_pos.parquet')
warm_users = set(pos['user_id'].unique())

ranked['is_cold_user'] = (~ranked['user_id'].isin(warm_users)).astype(int)
warm_scores = ranked[ranked['is_cold_user']==0]['lgbm_score']
cold_scores = ranked[ranked['is_cold_user']==1]['lgbm_score']
print(f'Warm preds: {len(warm_scores):,}  Cold preds: {len(cold_scores):,}')

plt.figure(figsize=(11, 6))
plt.hist(warm_scores, bins=80, alpha=0.55, label=f'Warm users (n={len(warm_scores):,})',
         density=True, color='steelblue')
plt.hist(cold_scores, bins=80, alpha=0.55, label=f'Cold users (n={len(cold_scores):,})',
         density=True, color='darkorange')
plt.xlabel('LightGBM Score'); plt.ylabel('Density')
plt.title('Score Distribution: Cold vs Warm Users\n(Model phân biệt 2 cohort nhờ is_cold_pair + NaN handling)',
          fontsize=12)
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(f'{FIGS_DIR}/03_score_distribution_cold_vs_warm.png', dpi=110, bbox_inches='tight')
plt.show()
print('Saved: 03_score_distribution_cold_vs_warm.png')

## Bonus — Feature Importance (gain)

In [ ]:
fi = pd.Series(model.feature_importance(importance_type='gain'),
               index=FEATURE_COLS).sort_values(ascending=True).tail(20)
plt.figure(figsize=(9, 8))
fi.plot.barh(color='teal')
plt.xlabel('Total Gain')
plt.title('LightGBM Feature Importance — Top 20')
plt.tight_layout()
plt.savefig(f'{FIGS_DIR}/04_feature_importance.png', dpi=110, bbox_inches='tight')
plt.show()
print('Saved: 04_feature_importance.png')
print('\nAll 4 figures ready in', FIGS_DIR)